# Duka Electronics Sales Exploration

**Period:** 1 January–30 June 2026  
**Currency:** Kenyan shillings (KES)

## Executive Summary

- Duka Electronics recorded 1,000 orders between January and June 2026.
- The highest-revenue branch was Nairobi CBD, generating KES 14,101,258.
- The top-revenue product was Laptop, contributing KES 18,280,437.
- Weekend orders totalled 302, compared with 698 weekday orders.
- Premium online orders from Nairobi and Mombasa numbered 117 and generated KES 9,588,486.

## 1. Load data and first-five-minutes inspection

The first five minutes should establish the dataset's size, structure, completeness, date coverage and key categories before analysis. The code below documents those checks.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA = Path("duka_sales.csv")
df = pd.read_csv(DATA, parse_dates=["order_date"])
print("Shape:", df.shape)
display(df.head())
display(df.dtypes.rename("dtype").to_frame())
display(df.isna().sum().rename("missing_rows").to_frame())
print("Date range:", df["order_date"].min(), "to", df["order_date"].max())
for col in ["branch", "city", "product", "channel"]:
    print(f"\\n{col} categories:")
    print(sorted(df[col].dropna().unique()))
print("Duplicate order IDs:", df["order_id"].duplicated().sum())

Shape: (1000, 8)


,order_id,order_date,branch,city,product,channel,units,unit_price
0,ORD00001,2026-04-02,Nairobi CBD,Nairobi,Smartphone,Online,1,35666
1,ORD00002,2026-05-26,Mombasa,Mombasa,Monitor,Online,2,23688
2,ORD00003,2026-02-15,Westlands,Nairobi,Tablet,In-store,2,34958
3,ORD00004,2026-01-18,Kisumu,Kisumu,Tablet,In-store,2,31195
4,ORD00005,2026-05-03,Kisumu,Kisumu,Smartphone,Online,1,50970


,dtype
order_id,object
order_date,datetime64[ns]
branch,object
city,object
product,object
channel,object
units,int64
unit_price,int64


,missing_rows
order_id,0
order_date,0
branch,0
city,0
product,0
channel,0
units,0
unit_price,0


Date range: 2026-01-01 00:00:00 to 2026-06-30 00:00:00
\nbranch categories:
['Kisumu', 'Mombasa', 'Nairobi CBD', 'Nakuru', 'Westlands']
\ncity categories:
['Kisumu', 'Mombasa', 'Nairobi', 'Nakuru']
\nproduct categories:
['Headphones', 'Keyboard', 'Laptop', 'Monitor', 'Mouse', 'Printer', 'Smartphone', 'Tablet']
\nchannel categories:
['In-store', 'Online']
Duplicate order IDs: 0


**Inspection notes:** The expected extract contains 1,000 orders and eight source columns. The checks above make the shape, data types, missingness, date range and categories visible. Confirm the printed results after regenerating the CSV; investigate any unexpected missing values, duplicate order IDs or dates outside January–June 2026 before proceeding.

## 2. Create derived columns

Revenue is units multiplied by unit price. `price_band` is based on unit price, using the assignment threshold of KES 28,000.

In [2]:
df["revenue"] = df["units"] * df["unit_price"]
df["month"] = df["order_date"].dt.month_name()
df["weekday"] = df["order_date"].dt.day_name()
df["is_weekend"] = df["order_date"].dt.dayofweek >= 5
df["price_band"] = np.where(df["unit_price"] >= 28000, "Premium", "Value")
display(df.head())

,order_id,order_date,branch,city,product,channel,units,unit_price,revenue,month,weekday,is_weekend,price_band
0,ORD00001,2026-04-02,Nairobi CBD,Nairobi,Smartphone,Online,1,35666,35666,April,Thursday,False,Premium
1,ORD00002,2026-05-26,Mombasa,Mombasa,Monitor,Online,2,23688,47376,May,Tuesday,False,Value
2,ORD00003,2026-02-15,Westlands,Nairobi,Tablet,In-store,2,34958,69916,February,Sunday,True,Premium
3,ORD00004,2026-01-18,Kisumu,Kisumu,Tablet,In-store,2,31195,62390,January,Sunday,True,Premium
4,ORD00005,2026-05-03,Kisumu,Kisumu,Smartphone,Online,1,50970,50970,May,Sunday,True,Premium


## Task 3 — Branch ranking by revenue and orders

The rankings are shown separately. They need not match: a branch can process many low-value orders while another processes fewer high-value orders.

In [3]:
branch_summary = (df.groupby("branch")
    .agg(total_revenue=("revenue", "sum"), orders=("order_id", "nunique"))
    .sort_values("total_revenue", ascending=False))
display(branch_summary)
print("Revenue ranking:", list(branch_summary.sort_values("total_revenue", ascending=False).index))
print("Order ranking:", list(branch_summary.sort_values("orders", ascending=False).index))

,total_revenue,orders
branch,,
Nairobi CBD,14101258,317
Westlands,11727687,216
Kisumu,9207192,168
Mombasa,8654044,167
Nakuru,7134310,132


Revenue ranking: ['Nairobi CBD', 'Westlands', 'Kisumu', 'Mombasa', 'Nakuru']
Order ranking: ['Nairobi CBD', 'Westlands', 'Kisumu', 'Mombasa', 'Nakuru']


## Task 4 — Top products by revenue and units sold

Revenue rewards products with high selling prices and/or volume. Units sold measures demand by quantity alone, so the lists can differ.

In [4]:
by_revenue = df.groupby("product")["revenue"].sum().sort_values(ascending=False)
by_units = df.groupby("product")["units"].sum().sort_values(ascending=False)
print("Top 3 by revenue")
display(by_revenue.head(3).to_frame("revenue_KES"))
print("Top 3 by units")
display(by_units.head(3).to_frame("units_sold"))

Top 3 by revenue


,revenue_KES
product,
Laptop,18280437
Smartphone,14574097
Tablet,6497717


Top 3 by units


,units_sold
product,
Smartphone,321
Headphones,253
Laptop,231


## Task 5 — Weekend versus weekday orders

Average order revenue is total revenue divided by the number of orders in each group.

In [5]:
weekend_comparison = (df.groupby("is_weekend")
    .agg(orders=("order_id", "nunique"),
         total_revenue=("revenue", "sum"),
         average_order_revenue=("revenue", "mean"))
    .rename(index={False: "Weekday", True: "Weekend"}))
display(weekend_comparison)

,orders,total_revenue,average_order_revenue
is_weekend,,,
Weekday,698,35193002,50419.773639
Weekend,302,15631489,51759.897351


## Task 6 — Best and worst month

The percentage difference below expresses how much higher the best month is than the worst month, using the worst month as the denominator.

In [6]:
monthly_revenue = df.groupby(df["order_date"].dt.month_name())["revenue"].sum()
month_order = ["January", "February", "March", "April", "May", "June"]
monthly_revenue = monthly_revenue.reindex(month_order)
best_month = monthly_revenue.idxmax()
worst_month = monthly_revenue.idxmin()
pct_difference = (monthly_revenue.max() - monthly_revenue.min()) / monthly_revenue.min() * 100
display(monthly_revenue.to_frame("revenue_KES"))
print(f"Best: {best_month}; worst: {worst_month}; difference: {pct_difference:.1f}%")

,revenue_KES
order_date,
January,9658203
February,6886286
March,8033788
April,8523398
May,9898503
June,7824313


Best: May; worst: February; difference: 43.7%


## Task 7 — Premium online orders in Nairobi and Mombasa

The filter uses unit price of at least KES 28,000, Online channel, and the two specified cities. The export deliberately uses `index=False`.

In [7]:
premium_online = df[
    (df["price_band"] == "Premium") &
    (df["channel"] == "Online") &
    (df["city"].isin(["Nairobi", "Mombasa"]))
].copy()
print("Orders:", len(premium_online))
print("Total revenue (KES):", premium_online["revenue"].sum())
premium_online.to_csv("outputs/premium_online_coast_capital.csv", index=False)
display(premium_online.head())

Orders: 117
Total revenue (KES): 9588486


,order_id,order_date,branch,city,product,channel,units,unit_price,revenue,month,weekday,is_weekend,price_band
0,ORD00001,2026-04-02,Nairobi CBD,Nairobi,Smartphone,Online,1,35666,35666,April,Thursday,False,Premium
7,ORD00008,2026-04-15,Nairobi CBD,Nairobi,Smartphone,Online,1,45414,45414,April,Wednesday,False,Premium
8,ORD00009,2026-03-29,Mombasa,Mombasa,Monitor,Online,1,41518,41518,March,Sunday,True,Premium
18,ORD00019,2026-03-29,Nairobi CBD,Nairobi,Smartphone,Online,1,59535,59535,March,Sunday,True,Premium
23,ORD00024,2026-02-25,Mombasa,Mombasa,Laptop,Online,1,76926,76926,February,Wednesday,False,Premium


## Task 8 — Excel reconciliation

Open `excel/duka_branch_summary.xlsx`. The Branch Summary sheet uses a `SUMIFS`-based revenue calculation and `XLOOKUP` for region. The PivotTable Revenue sheet is a formula-driven branch-by-channel cross-tab. Its row and grand totals should equal the pandas branch and channel totals below. (If your assessor specifically requires an Excel-native PivotTable object, create one from the Sales Data table in Excel using Insert → PivotTable; the workbook includes the source table.)

In [8]:
pandas_total = df["revenue"].sum()
print("Pandas grand total (KES):", pandas_total)
print("Pandas branch totals:")
display(df.groupby("branch")["revenue"].sum().sort_values(ascending=False).to_frame())
print("Pandas branch x channel totals:")
display(pd.pivot_table(df, index="branch", columns="channel",
                       values="revenue", aggfunc="sum", margins=True))
print("Excel workbook supplied: excel/duka_branch_summary.xlsx")

Pandas grand total (KES): 50824491
Pandas branch totals:


,revenue
branch,
Nairobi CBD,14101258
Westlands,11727687
Kisumu,9207192
Mombasa,8654044
Nakuru,7134310


Pandas branch x channel totals:


channel,In-store,Online,All
branch,,,
Kisumu,5718678,3488514,9207192
Mombasa,5611113,3042931,8654044
Nairobi CBD,9372387,4728871,14101258
Nakuru,4579826,2554484,7134310
Westlands,8091634,3636053,11727687
All,33373638,17450853,50824491


Excel workbook supplied: excel/duka_branch_summary.xlsx


## Business interpretation

Use the tables above to explain what is driving performance rather than merely restating totals. Compare revenue and order rankings to distinguish volume from value; compare product revenue with units to identify high-value products versus fast movers; and compare weekend and weekday average order values before recommending staffing or promotions. Treat the synthetic data as a training extract, not as audited company financial records.